# S6E9 V3：五折训练与可选融合

用户报告当前 Public LB：**0.94555**。本版本仅生成代码并通过语法/结构检查，**未执行训练**，不保证达到 0.94658。

默认：已验证特征的 CatBoost 五折。可选：LightGBM + 内层五折目标编码 + 50/50 概率融合；这个分支尚未验证提分。

依赖：`numpy pandas scikit-learn catboost`。如启用 LightGBM，在你的内核环境自行安装 `lightgbm`；此 notebook 不自动安装软件。

原项目数据：`/Users/yw/unsw/Kaggle/data`。结果保存到原项目 `outputs/v3_时间戳`，每次独立目录。

编码思路参考：[公开收入编码实验](https://www.kaggle.com/competitions/playground-series-s6e9/discussion/742385)。代码为独立实现，正式效果以本地 OOF 和实际提交为准。

## 1. 定义训练函数
运行这一格仅加载依赖和定义函数，不开始训练。

In [1]:
"""S6E9: fixed-fold CatBoost; optional LightGBM with cross-fitted target encoding.

Default: python s6e9_train_v3.py
Optional: python s6e9_train_v3.py --models catboost lightgbm
Dependencies: numpy pandas scikit-learn catboost; optional lightgbm.
No training occurs when this module is imported.
"""
import argparse
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

TARGET = "Will_Buy_EV"
NUMERIC = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
           "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
           "Environmental_Concern_Level"]


def features(frame, model_name):
    """Row-local features only: no labels or full-data target statistics."""
    x = frame.drop(columns=["id", TARGET], errors="ignore").copy()
    for col, scale, positions in [("Age", 1, 2), ("Annual_Income_USD", 1, 6),
                                 ("Daily_Commute_km", 10, 3),
                                 ("Charging_Stations_Near_Home", 1, 2),
                                 ("Charging_Stations_Near_Work", 1, 2)]:
        values = np.rint(frame[col].to_numpy() * scale).astype(np.int64)
        for digit in range(positions):
            x[f"{col}_digit_{digit}"] = (values // 10**digit % 10).astype(np.int8)
    income = np.rint(frame["Annual_Income_USD"]).astype(np.int64)
    x["income_last_2"] = income % 100
    x["income_last_3"] = income % 1000
    if model_name == "catboost":
        for col in NUMERIC:
            scale = 10 if col == "Daily_Commute_km" else 1
            x[f"{col}_as_category"] = np.rint(frame[col] * scale).astype(np.int64).astype(str)
    else:
        x["income_prefix_100"] = income // 100
        x["income_prefix_1000"] = income // 1000
    return x


def encode_fold(train_x, train_y, valid_x, test_x, seed, smoothing=1.0):
    """Inner OOF for training TE; outer-training-only maps for validation/test."""
    a, b, c = [d.copy().reset_index(drop=True) for d in (train_x, valid_x, test_x)]
    labels = np.asarray(train_y)
    inner = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(a, labels))
    cats = a.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    keys = NUMERIC + cats + ["income_prefix_100", "income_prefix_1000"]
    for col in keys:
        encoded = np.full(len(a), np.nan)
        for fit_idx, hold_idx in inner:
            subset = pd.DataFrame({"key": a[col].iloc[fit_idx].to_numpy(), "y": labels[fit_idx]})
            stats = subset.groupby("key")["y"].agg(["sum", "count"])
            prior = labels[fit_idx].mean()
            mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
            encoded[hold_idx] = a[col].iloc[hold_idx].map(mapping).fillna(prior).to_numpy()
        stats = pd.DataFrame({"key": a[col], "y": labels}).groupby("key")["y"].agg(["sum", "count"])
        prior = labels.mean()
        mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
        a[f"{col}_te"] = encoded
        b[f"{col}_te"] = b[col].map(mapping).fillna(prior).to_numpy()
        c[f"{col}_te"] = c[col].map(mapping).fillna(prior).to_numpy()
        assert np.isfinite(encoded).all()
    # Label-free frequency: exclude a training row's own count contribution.
    counts = a["Annual_Income_USD"].value_counts()
    a["income_frequency"] = (a["Annual_Income_USD"].map(counts) - 1) / (len(a) - 1)
    for frame in (b, c):
        frame["income_frequency"] = frame["Annual_Income_USD"].map(counts).fillna(0) / len(a)
    for col in cats:
        dtype = pd.CategoricalDtype(categories=pd.unique(a[col]))
        for frame in (a, b, c):
            frame[col] = frame[col].astype(dtype)
    return a, b, c


def run(project="/Users/yw/unsw/Kaggle", models=("catboost",), threads=-1):
    models = tuple(models)
    if not models or len(set(models)) != len(models) or not set(models) <= {"catboost", "lightgbm"}:
        raise ValueError("models must contain distinct choices from catboost/lightgbm")
    # Check requested dependencies before starting any expensive training.
    if "catboost" in models:
        from catboost import CatBoostClassifier
    if "lightgbm" in models:
        try:
            import lightgbm as lgb
        except ImportError as error:
            raise ImportError("Optional LightGBM is unavailable; install lightgbm or run CatBoost only.") from error

    project = Path(project)
    train = pd.read_csv(project / "data/train.csv")
    test = pd.read_csv(project / "data/test.csv")
    assert train.drop(columns=TARGET).columns.tolist() == test.columns.tolist()
    assert train.id.is_unique and test.id.is_unique and not train.id.isin(test.id).any()
    assert not train.isna().any().any() and not test.isna().any().any()
    assert set(train[TARGET].unique()) == {"No", "Yes"}
    for frame in (train, test):
        assert np.allclose(frame.Annual_Income_USD, frame.Annual_Income_USD.round())
        assert np.allclose(frame.Daily_Commute_km * 10, (frame.Daily_Commute_km * 10).round())
    y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")
    folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(train, y))
    out = project / "outputs" / ("v3_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=False)
    (out / "config.json").write_text(json.dumps({
        "models": models, "folds": 5, "seed": 42, "threads": threads,
        "train_rows": len(train), "test_rows": len(test),
        "note": "LightGBM/encoding and blend are unvalidated candidates; compare their OOF results.",
    }, indent=2))
    print("Output directory:", out, flush=True)
    all_oof, all_test, records = {}, {}, []

    def save_submission(name, probabilities):
        assert len(probabilities) == len(test)
        assert np.isfinite(probabilities).all()
        assert ((probabilities >= 0) & (probabilities <= 1)).all()
        pd.DataFrame({"id": test.id.to_numpy(), TARGET: probabilities}).to_csv(
            out / f"submission_{name}.csv", index=False)

    for name in models:
        x, xt = features(train, name), features(test, name)
        assert x.columns.tolist() == xt.columns.tolist()
        cats = x.select_dtypes(include=["object", "string", "category"]).columns.tolist()
        oof = np.full(len(train), np.nan)
        coverage = np.zeros(len(train), dtype=np.int8)
        test_sum = np.zeros(len(test))
        for fold, (ti, vi) in enumerate(folds, start=1):
            print(f"\n========== {name} | Fold {fold}/5 ==========", flush=True)
            a, b, c = x.iloc[ti], x.iloc[vi], xt
            if name == "catboost":
                # Same features/parameters as the successful single-fold candidate.
                model = CatBoostClassifier(
                    iterations=2000, learning_rate=0.03, depth=7,
                    loss_function="Logloss", eval_metric="AUC", random_seed=41 + fold,
                    thread_count=threads, allow_writing_files=False, verbose=400,
                )
                model.fit(a, y.iloc[ti], cat_features=cats, eval_set=(b, y.iloc[vi]),
                          early_stopping_rounds=150, use_best_model=True)
                model.save_model(str(out / f"{name}_fold{fold}.cbm"))
                best = int(model.get_best_iteration()) + 1
            else:
                a, b, c = encode_fold(a, y.iloc[ti], b, c, seed=1041 + fold)
                bins = [1023 if col == "Annual_Income_USD" else 255 for col in a.columns]
                model = lgb.LGBMClassifier(
                    n_estimators=5000, learning_rate=0.05, max_depth=5, num_leaves=31,
                    min_child_samples=100, reg_lambda=5.0, max_bin_by_feature=bins,
                    objective="binary", metric="auc", random_state=41 + fold, n_jobs=threads,
                    verbosity=-1, deterministic=True, force_col_wise=True,
                )
                model.fit(a, y.iloc[ti], eval_set=[(b, y.iloc[vi])], eval_metric="auc",
                          callbacks=[lgb.early_stopping(200, first_metric_only=True), lgb.log_evaluation(400)])
                model.booster_.save_model(str(out / f"{name}_fold{fold}.txt"))
                best = int(model.best_iteration_)
                # Persist fold-specific TE maps' source indices, for reproducible later inference.
                np.save(out / f"{name}_fold{fold}_train_indices.npy", ti)
            assert np.array_equal(model.classes_, [0, 1])
            vp = model.predict_proba(b)[:, 1]
            tp = model.predict_proba(c)[:, 1]
            assert np.isfinite(vp).all() and np.isfinite(tp).all()
            oof[vi], coverage[vi] = vp, coverage[vi] + 1
            test_sum += tp
            auc = float(roc_auc_score(y.iloc[vi], vp))
            records.append({"model": name, "fold": fold, "auc": auc, "best_trees": best})
            np.savez_compressed(out / f"{name}_fold{fold}_predictions.npz",
                                valid_indices=vi, valid_ids=train.id.iloc[vi].to_numpy(),
                                valid_prediction=vp, test_ids=test.id.to_numpy(), test_prediction=tp)
            pd.DataFrame(records).to_csv(out / "fold_scores.csv", index=False)
            print("Fold AUC:", auc, flush=True)
        assert (coverage == 1).all() and np.isfinite(oof).all()
        all_oof[name], all_test[name] = oof, test_sum / len(folds)
        pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(), "prediction": oof}).to_csv(
            out / f"oof_{name}.csv", index=False)
        save_submission(name, all_test[name])
        print(name, "OOF AUC:", roc_auc_score(y, oof), flush=True)

    if len(models) == 2:
        # One predeclared blend, avoiding an extensive weight search on the same OOF.
        all_oof["blend_50_50"] = (all_oof["catboost"] + all_oof["lightgbm"]) / 2
        all_test["blend_50_50"] = (all_test["catboost"] + all_test["lightgbm"]) / 2
        save_submission("blend_50_50", all_test["blend_50_50"])
        pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(),
                      "prediction": all_oof["blend_50_50"]}).to_csv(out / "oof_blend_50_50.csv", index=False)
    scores = {name: float(roc_auc_score(y, prediction)) for name, prediction in all_oof.items()}
    best_name = max(scores, key=scores.get)
    save_submission("best_oof", all_test[best_name])
    (out / "summary.json").write_text(json.dumps({
        "oof_auc": scores, "selected": best_name,
        "selection_note": "Selected on development OOF; this is not an independent audit or a Kaggle score.",
    }, indent=2))
    print("\nOOF comparison:", scores)
    print("Selected:", best_name, "\nCandidate:", out / "submission_best_oof.csv")
    return out




## 2. 手动开始训练

默认只运行 CatBoost 五折。想运行两个模型和融合，将 `MODELS` 改为 `("catboost", "lightgbm")`。

启用两个模型时会先检查 LightGBM 是否安装。程序保存每折预测和模型，但未实现自动断点续训；重新运行会新建目录并重新训练。

输出包含各模型 OOF、各模型提交、可选 50/50 融合，以及按 OOF 选择的 `submission_best_oof.csv`。OOF 用于开发与选择，不等同于独立审计或排行榜分数。

In [2]:
PROJECT = "/Users/yw/unsw/Kaggle"
MODELS = ("catboost",)  # 可选：( "catboost", "lightgbm" )
THREADS = -1

result_dir = run(project=PROJECT, models=MODELS, threads=THREADS)

Output directory: /Users/yw/unsw/Kaggle/outputs/v3_20260925_180457_278867

========== catboost | Fold 1/5 ==========
0:	test: 0.9314015	best: 0.9314015 (0)	total: 306ms	remaining: 10m 10s
400:	test: 0.9440926	best: 0.9440926 (400)	total: 1m 22s	remaining: 5m 30s
800:	test: 0.9444963	best: 0.9444967 (798)	total: 3m	remaining: 4m 29s
1200:	test: 0.9446189	best: 0.9446189 (1200)	total: 4m 54s	remaining: 3m 16s
1600:	test: 0.9446649	best: 0.9446651 (1584)	total: 6m 58s	remaining: 1m 44s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9446805786
bestIteration = 1732

Shrink model to first 1733 iterations.
Fold AUC: 0.9446805785652735

========== catboost | Fold 2/5 ==========
0:	test: 0.9346796	best: 0.9346796 (0)	total: 234ms	remaining: 7m 47s
400:	test: 0.9447678	best: 0.9447678 (400)	total: 1m 37s	remaining: 6m 29s
800:	test: 0.9451722	best: 0.9451722 (800)	total: 3m 21s	remaining: 5m 1s
1200:	test: 0.9452677	best: 0.9452677 (1200)	total: 5m 9s	remaining: 3m 26s
1600